In [186]:
from langchain.tools import BaseTool , tool
from typing import Type
from pydantic import BaseModel , Field

In [187]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence


load_dotenv()

llm = HuggingFaceEndpoint(
    # repo_id = "google/gemma-2-2b-it",
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    huggingfacehub_api_token="YOUR_HF_TOKEN",
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)



In [188]:
class MultiplyInput(BaseModel):
    a : int = Field(required = True , description='the first number to multiply')
    b : int = Field(required = True , description='the second number to multiply')


In [189]:
class MultiplyTool(BaseTool):
    name : str = "multiply"

    description : str = "multiply 2 numbers"

    args_schema : Type[BaseModel] = MultiplyInput

    def _run(self, a : int , b : int) -> int:
        return a * b
        

In [190]:
mutiply_tool = MultiplyTool()

In [191]:
result = mutiply_tool.invoke({'a' : 3 , 'b' : 4})

In [192]:
result

12

In [193]:
#  TOOLKit

In [194]:
@tool
def add(a : int , b : int) -> int:
    """Add 2 numbers"""
    return a + b

@tool 
def multiply(a : int , b : int) -> int:
    """Multiply 2 numbers"""
    return a * b

In [195]:
class MathToolkit :
    def get_tools(self):
        return [add , multiply]

In [196]:
toolkit = MathToolkit()

In [197]:
tools = toolkit.get_tools()

In [198]:
for tool in tools :
    print(tool)

name='add' description='Add 2 numbers' args_schema=<class 'langchain_core.utils.pydantic.add'> func=<function add at 0x7fcd147a13a0>
name='multiply' description='Multiply 2 numbers' args_schema=<class 'langchain_core.utils.pydantic.multiply'> func=<function multiply at 0x7fcd1475fce0>


In [199]:
# binding the model with tools

llm_with_tools = model.bind_tools([multiply])

In [200]:
llm_with_tools.invoke("hi how are you")
# gives us normal response

AIMessage(content="<think>\nWe are given a tool to multiply two integers. The user's message is a greeting and does not require a tool call.\n Since the user is just greeting, we should respond appropriately without using any tool.\n</think>\nI'm just a virtual assistant, so I don't have feelings, but I'm fully operational and ready to help! 😊 How can I assist you today? Whether it's calculations, information, or anything else—just let me know!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 160, 'total_tokens': 255}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--0ebfd2cb-5151-41fc-9193-f2dfc37d43d9-0', usage_metadata={'input_tokens': 160, 'output_tokens': 95, 'total_tokens': 255})

In [201]:
# TOOL CALLING
llm_with_tools.invoke("multiply 3 with 4")
# suggesting to use a tool 


AIMessage(content='<think>\nWe are given a function named "multiply" that takes two integer parameters: a and b.\n The user request is to "multiply 3 with 4". Therefore, we need to set a=3 and b=4.\n Then, we call the function with these arguments.\n</think>\nI\'ll multiply 3 and 4 for you using the multiply function:\n\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 4}', 'name': 'multiply', 'description': None}, 'id': 'call_p3nyYgxy8kGEhlKbCGpMfHyD', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 163, 'total_tokens': 266}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--fa34e709-64d1-43d8-81c2-6c2de657170d-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'call_p3nyYgxy8kGEhlKbCGpMfHyD', 'type': 'tool_call'}], usage_metadata={'input_tokens': 163, 'output_tokens': 103, 'total_tokens': 266})

In [202]:
llm_with_tools.invoke("multiply 3 with 4").tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 4},
  'id': 'call_suBK6wCPwta2wqa5kGJUgyNT',
  'type': 'tool_call'}]

In [203]:
llm_with_tools.invoke("multiply 3 with 4").tool_calls[0]
# llm recognised that this tool will be helpful to solve this query
# it never uses that tool itself by giving the tool input by itself , LLM will just suggest us that for this input we can use a tool , and lets us know the schema 

{'name': 'multiply',
 'args': {'a': 3, 'b': 4},
 'id': 'call_YVOtJA2UJEpHXcRA0SF9acEd',
 'type': 'tool_call'}

In [204]:
# TOOL EXECUTION
# LLM has told us what to send as input for this tool to work
# llm_with_tools.invoke("multiply 3 with 4").tool_calls[0].args

llm_with_tools.invoke("multiply 3 with 4").tool_calls[0]['args']


{'a': 3, 'b': 4}

In [205]:
multiply.invoke(
llm_with_tools.invoke("multiply 3 with 4").tool_calls[0]['args'])

12

In [206]:
# what if we just send the whole tool call???

tool_call = llm_with_tools.invoke("multiply 3 with 4").tool_calls[0]

In [207]:
multiply.invoke(tool_call)
# now we can send this ToolMessage to LLM
# our flow is , HumanMessage to LLM --> LLM suggests use a tool for this --> we use the LLM suggested input for the tool and get the ToolMessage --> we send the ToolMessage to the LLM

ToolMessage(content='12', name='multiply', tool_call_id='call_3riRrvn6f7wGzR0AC4niaCk1')

In [209]:
# RECREATING THE WORKFLOW
from langchain_core.messages import HumanMessage

query = HumanMessage("multiply 3 with 4")

# we will keep a message history

message = [query]

In [211]:
result = llm_with_tools.invoke("multiply 3 with 4")

In [213]:
result
# llm is suggesting a tool now

message.append(result)

In [216]:
multiply.invoke(result.tool_calls[0])
# now we got the ToolMessage

message.append(multiply.invoke(result.tool_calls[0]))

In [218]:
message
# now we have the whole conversational history

[HumanMessage(content='multiply 3 with 4', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<think>\nWe are given the function "multiply" which takes two integers and returns their product.\n The user asked to multiply 3 with 4.\n Therefore, we will call the function with a=3 and b=4.\n</think>\nI\'ll multiply 3 and 4 for you using the multiply function:\n\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 4}', 'name': 'multiply', 'description': None}, 'id': 'call_EvYkLwOi5lLbXwaUaZZLxStW', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 163, 'total_tokens': 253}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--8aa05c9e-c6d4-4f86-a714-d293e2279d14-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'call_EvYkLwOi5lLbXwaUaZZLxStW', 'type': 'tool_call'}], usage_metadata={'input_tokens': 163, 

In [221]:
llm_with_tools.invoke(message)
# we sent the whole context to the llm

AIMessage(content='<think>\nWe are given a request to multiply 3 with 4.\n We have a function called "multiply" that takes two parameters: a and b.\n We will call the function with a=3 and b=4.\n The expected result is 3 * 4 = 12.\n We return the result as a string in a JSON-like format as per the example.\n</think>\nThe result of multiplying 3 by 4 is **12**.  \n\n### Calculation:\n```\n3 × 4 = 12\n```', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 41, 'total_tokens': 148}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--aad8c191-fe51-4d69-b867-7fd513656fd2-0', usage_metadata={'input_tokens': 41, 'output_tokens': 107, 'total_tokens': 148})